# Task 2: Bag of Words and TF-IDF features

Build representations from the shared seed-46 split. The input joins `Summary` and `Text`; `Score` is not a feature. Fit vocabulary and TF-IDF weights on training data only, then transform validation and test data.


In [1]:
from pathlib import Path
import sys
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Works when launched from the repository root or notebooks/.
root = Path.cwd() if (Path.cwd() / "data_utils.py").exists() else Path.cwd().parent
if not (root / "data_utils.py").exists():
    raise FileNotFoundError("Run this notebook from the repository root or notebooks directory")
sys.path.insert(0, str(root))
from data_utils import load_splits

train, val, test = load_splits()
print(f"Rows: train={len(train):,}, validation={len(val):,}, test={len(test):,}")
assert [len(train), len(val), len(test)] == [30123, 6952, 9269]
assert not (set(train.Id) & set(val.Id) or set(train.Id) & set(test.Id) or set(val.Id) & set(test.Id))


Rows: train=30,123, validation=6,952, test=9,269


## Fit on training text only

Both vectorizers use scikit-learn defaults: lowercase text, single-word tokens of at least two word characters, and no stop-word removal or vocabulary size limit. Bag of Words records counts; TF-IDF uses smoothed inverse document frequency and L2 row normalization.


In [2]:
bow = CountVectorizer()
tfidf = TfidfVectorizer()

X_train_bow = bow.fit_transform(train["text"])
X_val_bow = bow.transform(val["text"])
X_test_bow = bow.transform(test["text"])

X_train_tfidf = tfidf.fit_transform(train["text"])
X_val_tfidf = tfidf.transform(val["text"])
X_test_tfidf = tfidf.transform(test["text"])

for name, vectorizer, matrices in [
    ("Bag of Words", bow, (X_train_bow, X_val_bow, X_test_bow)),
    ("TF-IDF", tfidf, (X_train_tfidf, X_val_tfidf, X_test_tfidf)),
]:
    size = len(vectorizer.get_feature_names_out())
    assert all(matrix.shape[1] == size for matrix in matrices)
    print(f"{name}: vocabulary={size:,}; train={matrices[0].shape}, validation={matrices[1].shape}, test={matrices[2].shape}")


Bag of Words: vocabulary=35,788; train=(30123, 35788), validation=(6952, 35788), test=(9269, 35788)
TF-IDF: vocabulary=35,788; train=(30123, 35788), validation=(6952, 35788), test=(9269, 35788)
